# Efficient Frontier, Tangency Portfolio, Sharpe Ratio

Add a target return to the minimum-variance problem and the whole efficient frontier drops out
in closed form. Add a risk-free asset and the frontier collapses to a straight line through one
special portfolio: the one with the highest Sharpe ratio.

## 0. Math link

- **Solving linear systems**: every formula below needs $\Sigma^{-1}b$, computed by solving
  $\Sigma x = b$. [LU decomposition](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/051_lu_decomposition.ipynb)
  is how `np.linalg.solve` does it; [Matrix inverse](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/02_matrices/026_matrix_inverse.ipynb)
  explains why we avoid forming $\Sigma^{-1}$ explicitly.
- **Lagrange multipliers**, now with two constraints: introduced in
  [03 Global minimum-variance portfolio](./03_global_minimum_variance_portfolio.ipynb).
- $w^\top\mu$ and $w^\top\Sigma w$ from [01 Portfolio variance](./01_portfolio_variance.ipynb).

## 1. Introduction

03/04 drew the frontier for two assets by trying every weight. 04/03 found its leftmost point
for $n$ assets. This notebook finds **every** point: for each target return, the portfolio with
the lowest variance.

Then it adds cash. With a risk-free rate $r_f$, an investor can mix any risky portfolio with
cash, and the best risky portfolio to mix is the one with the steepest line from $r_f$: the
**tangency portfolio**, the portfolio with the highest **Sharpe ratio**.

After this notebook you can:

- compute the efficient frontier in closed form, and know why it is a hyperbola;
- explain two-fund separation;
- compute the tangency portfolio and its Sharpe ratio, and draw the capital market line;
- see why the tangency weights are fragile, and why that matters (module 08).

## 2. Theory

### The frontier problem

For a target return $m$:

$$
\min_w\ \tfrac12\,w^\top\Sigma w
\quad\text{subject to}\quad
w^\top\mu = m,\qquad \mathbf{1}^\top w = 1
$$

Two constraints, two multipliers:

$$
\mathcal{L} = \tfrac12\,w^\top\Sigma w - \lambda\,(w^\top\mu - m) - \gamma\,(\mathbf{1}^\top w - 1)
$$

$$
\nabla_w\mathcal{L} = \Sigma w - \lambda\mu - \gamma\mathbf{1} = 0
\quad\Longrightarrow\quad
w = \lambda\,\Sigma^{-1}\mu + \gamma\,\Sigma^{-1}\mathbf{1}
$$

### Four scalars do all the work

Define

$$
A = \mathbf{1}^\top\Sigma^{-1}\mathbf{1},\qquad
B = \mathbf{1}^\top\Sigma^{-1}\mu,\qquad
C = \mu^\top\Sigma^{-1}\mu,\qquad
D = AC - B^2
$$

($A, C > 0$ for positive definite $\Sigma$, and $D > 0$ unless all expected returns are equal.)
Plugging $w$ into the two constraints gives a $2 \times 2$ linear system:

$$
\begin{pmatrix} C & B \\ B & A \end{pmatrix}
\begin{pmatrix} \lambda \\ \gamma \end{pmatrix}
= \begin{pmatrix} m \\ 1 \end{pmatrix}
\quad\Longrightarrow\quad
\lambda = \frac{Am - B}{D},\qquad \gamma = \frac{C - Bm}{D}
$$

### The frontier is a hyperbola

$w^\top\Sigma w = w^\top(\lambda\mu + \gamma\mathbf{1}) = \lambda m + \gamma$, so

$$
\sigma^2(m) = \frac{A m^2 - 2Bm + C}{D}
$$

Quadratic in $m$: a hyperbola in the $(\sigma, \mu)$ plane, as in 03/04. Its minimum is at
$m = B/A$ with $\sigma^2 = 1/A$: the GMV portfolio of 04/03. The **efficient** frontier is the
upper branch, $m \ge B/A$.

### Two-fund separation

$w(m)$ is linear in $m$. Any two distinct frontier portfolios therefore generate the entire
frontier: every frontier portfolio is a (possibly leveraged) combination of them. A convenient
pair is the GMV portfolio $\Sigma^{-1}\mathbf{1}/A$ and $\Sigma^{-1}\mu/B$.

### Adding a risk-free asset

Hold weights $w$ in the risky assets and $1 - \mathbf{1}^\top w$ in cash at rate $r_f$ (no
budget constraint on $w$). Expected return $r_f + w^\top(\mu - r_f\mathbf{1})$, variance
$w^\top\Sigma w$. Minimizing variance for a target excess return gives

$$
\Sigma w = \lambda\,(\mu - r_f\mathbf{1})
\quad\Longrightarrow\quad
w \propto \Sigma^{-1}(\mu - r_f\mathbf{1})
$$

Every efficient portfolio holds risky assets in the **same proportions**; only the amount of
cash changes. Scaling to a fully invested risky portfolio:

$$
\boxed{\,w_{\text{tan}} = \frac{\Sigma^{-1}(\mu - r_f\mathbf{1})}{\mathbf{1}^\top\Sigma^{-1}(\mu - r_f\mathbf{1})}\,}
$$

This requires $\mathbf{1}^\top\Sigma^{-1}(\mu - r_f\mathbf{1}) > 0$, which holds when
$r_f < B/A$ (the risk-free rate below the GMV return).

### The Sharpe ratio

$$
SR_p = \frac{\mu_p - r_f}{\sigma_p}
$$

the slope of the line from $(0, r_f)$ to the portfolio in the $(\sigma, \mu)$ plane. The
tangency portfolio maximizes it, with

$$
SR_{\text{tan}}^2 = (\mu - r_f\mathbf{1})^\top\,\Sigma^{-1}\,(\mu - r_f\mathbf{1})
$$

### The capital market line

Mixing the tangency portfolio (weight $a$) with cash ($1-a$) gives
$\mu_p = r_f + a(\mu_{\text{tan}} - r_f)$ and $\sigma_p = a\,\sigma_{\text{tan}}$, so

$$
\mu_p = r_f + SR_{\text{tan}}\,\sigma_p
$$

a straight line, tangent to the risky frontier at $w_{\text{tan}}$. $a < 1$ lends cash,
$a > 1$ borrows (leverage). Every point on it beats the risky frontier at the same volatility.

### The first-order condition, read as CAPM

$\Sigma w_{\text{tan}} \propto \mu - r_f\mathbf{1}$ says, row by row, that each asset's excess
return is proportional to its covariance with the tangency portfolio. Normalizing:

$$
\mu_i - r_f = \beta_i\,(\mu_{\text{tan}} - r_f),
\qquad
\beta_i = \frac{\operatorname{Cov}(R_i, R_{\text{tan}})}{\sigma_{\text{tan}}^2}
$$

This is pure algebra about the tangency portfolio. CAPM (04/05) adds the economics: in
equilibrium, the tangency portfolio is the market portfolio.

### Units

Sharpe ratios carry the periodicity of the returns. With independent periods,
$SR_{\text{annual}} = SR_{\text{period}} \cdot \sqrt{\text{periods\_per\_year}}$ (mean scales
with $T$, volatility with $\sqrt T$).

## 3. Worked Examples

### Example 1: two uncorrelated assets

A: $\mu = 7\%$, $\sigma = 16\%$. B: $\mu = 2\%$, $\sigma = 5\%$. $\rho = 0$, $r_f = 0.5\%$.

With diagonal $\Sigma$, $\Sigma^{-1}(\mu - r_f\mathbf{1})$ has entries $(\mu_i - r_f)/\sigma_i^2$:

$$
\frac{0.065}{0.0256} = 2.54,\qquad \frac{0.015}{0.0025} = 6.00
\quad\Longrightarrow\quad
w_{\text{tan}} = \Big(\tfrac{2.54}{8.54},\ \tfrac{6.00}{8.54}\Big) = (29.7\%,\ 70.3\%)
$$

$$
SR_{\text{tan}} = \sqrt{\frac{0.065^2}{0.0256} + \frac{0.015^2}{0.0025}} = \sqrt{0.165 + 0.090} = 0.505
$$

The single assets have Sharpe ratios $0.406$ and $0.300$. Combining them beats both:
for uncorrelated assets, squared Sharpe ratios add.

### Example 2: three asset classes

The 04/01 universe (Swiss equities, global equities, Swiss bonds) with expected returns
6.5%, 7% and 2%, and $r_f = 0.5\%$. The scalars (computed in Python):
$A = 435.2$, $B = 10.37$, $C = 0.333$, $D = 37.48$.

- GMV: $\mu = B/A = 2.38\%$, $\sigma = 1/\sqrt{A} = 4.79\%$ (as in 04/03).
- Frontier at $m = 5\%$: $\sigma = \sqrt{(A m^2 - 2Bm + C)/D} = 10.12\%$.
- Tangency: weights $(8.5\%,\ 21.0\%,\ 70.5\%)$, $\mu = 3.43\%$, $\sigma = 5.98\%$, $SR = 0.49$.
  The best single asset has $SR = 0.38$.

### Example 3: the CML beats the risky frontier

Target 5% expected return. On the risky frontier: 10.12% volatility. On the CML: hold
$a = (5 - 0.5)/(3.43 - 0.5) = 1.534$ in the tangency portfolio, borrowing 53% at $r_f$:

$$
\sigma = 1.534 \cdot 5.98\% = 9.17\%
$$

Same expected return, about 1 percentage point less volatility, if borrowing at $r_f$ is possible.

## 4. Finance Application

### Asset allocation in one formula

With a risk-aversion coefficient $\gamma$ (higher = more cautious), the mean-variance investor
holds $w = \frac{1}{\gamma}\Sigma^{-1}(\mu - r_f\mathbf{1})$ in risky assets and the rest in cash.
Risk aversion changes **how much** risk, not **which** risky portfolio. This is the formal
version of "conservative, balanced, growth": same building blocks, different cash share.

### The Sharpe ratio as a yardstick

The Sharpe ratio is the standard measure of risk-adjusted performance for funds and strategies.
Its appeal comes from the CML: if you can lend and borrow, the portfolio with the higher
Sharpe ratio can be levered to beat the other at **any** risk level.

Typical magnitudes: broad equity markets have long-run Sharpe ratios around 0.3–0.5; a
sustained Sharpe ratio above 1 for an unlevered long-only fund is rare.

### Where the theory meets practice

- **Borrowing is not at $r_f$.** With a higher borrowing rate the CML kinks, and investors who
  want more risk move along the risky frontier instead.
- **Constraints.** Long-only, position limits and regulatory rules give a frontier that lies
  inside the unconstrained one (see the chart below).
- **Inputs.** $w_{\text{tan}}$ depends on $\Sigma^{-1}\mu$. Small changes in expected returns,
  which are estimated badly (module 06), cause large changes in weights. The Python section
  shifts one expected return by 0.5 percentage points. Module 08/01 calls this "error
  maximization".

### Sharpe ratio pitfalls

- **Non-normal returns.** Strategies that sell options or credit protection show high Sharpe
  ratios until the tail event.
- **Smoothed returns.** Illiquid assets valued by appraisal (private equity, real estate) have
  artificially low volatility and inflated Sharpe ratios.
- **Annualization.** $\sqrt{12}$ and $\sqrt{252}$ assume independent periods; autocorrelation
  breaks this.
- **Estimation noise.** A Sharpe ratio estimated from a few years of data has a wide
  confidence interval (module 06/02).

### The usual misreading

"The tangency portfolio is the optimal portfolio." It is optimal **given** $\mu$, $\Sigma$ and
$r_f$, which are estimates. With the true inputs unknown, the estimated tangency portfolio is
often worse out-of-sample than naive alternatives such as equal weights.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(42)

In [ ]:
assets = ["Swiss equities", "Global equities", "Swiss bonds"]
vols = np.array([0.16, 0.17, 0.05])                       # annual, decimals
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
mu = np.array([0.065, 0.07, 0.02])                        # annual expected returns
rf = 0.005                                                # CHF risk-free rate, annual
Sigma = np.diag(vols) @ corr @ np.diag(vols)
ones = np.ones(len(mu))

### The scalars $A, B, C, D$

One call to `np.linalg.solve` with two right-hand sides: the LU factorization of $\Sigma$ is
computed once and reused.

In [ ]:
X = np.linalg.solve(Sigma, np.column_stack([ones, mu]))   # columns: Sigma^{-1} 1, Sigma^{-1} mu
Sigma_inv_1, Sigma_inv_mu = X[:, 0], X[:, 1]

A = ones @ Sigma_inv_1
B = ones @ Sigma_inv_mu
C = mu @ Sigma_inv_mu
D = A * C - B**2
print(f"A = {A:.2f}, B = {B:.4f}, C = {C:.4f}, D = {D:.3f}")
print(f"GMV: return {B / A:.2%}, volatility {np.sqrt(1 / A):.2%}")

### Frontier portfolios in closed form

In [ ]:
def frontier_weights(m):
    lam = (A * m - B) / D
    gam = (C - B * m) / D
    return lam * Sigma_inv_mu + gam * Sigma_inv_1


def frontier_vol(m):
    return np.sqrt((A * m**2 - 2 * B * m + C) / D)


m_target = 0.05
w_m = frontier_weights(m_target)
print(pd.Series(w_m, index=assets).map("{:+.2%}".format).to_string())
print(f"\nreturn {w_m @ mu:.2%}, sum {w_m.sum():.4f}")
print(f"volatility (w'Σw):     {np.sqrt(w_m @ Sigma @ w_m):.4%}")
print(f"volatility (formula):  {frontier_vol(m_target):.4%}")

# Same point from a numerical optimizer
cons = [{"type": "eq", "fun": lambda w: w.sum() - 1},
        {"type": "eq", "fun": lambda w: w @ mu - m_target}]
res = minimize(lambda w: w @ Sigma @ w, x0=np.full(3, 1 / 3), constraints=cons, method="SLSQP",
               options={"ftol": 1e-14})
print(f"SLSQP weights:         {np.round(res.x, 4)}")

### Two-fund separation

Build two frontier portfolios, mix them, and check the mix is on the frontier.

In [ ]:
w_gmv = Sigma_inv_1 / A
w_mu = Sigma_inv_mu / B

for t in [-0.5, 0.0, 0.5, 1.0, 1.5]:
    w_mix = (1 - t) * w_gmv + t * w_mu
    m_mix = w_mix @ mu
    print(f"t = {t:+.1f}: return {m_mix:6.2%}, vol {np.sqrt(w_mix @ Sigma @ w_mix):6.2%}, "
          f"frontier vol at that return {frontier_vol(m_mix):6.2%}")

### The tangency portfolio

In [ ]:
def tangency_weights(mu, Sigma, rf):
    x = np.linalg.solve(Sigma, mu - rf)
    return x / x.sum()


def sharpe(w, mu, Sigma, rf):
    return (w @ mu - rf) / np.sqrt(w @ Sigma @ w)


w_tan = tangency_weights(mu, Sigma, rf)
mu_tan, vol_tan = w_tan @ mu, np.sqrt(w_tan @ Sigma @ w_tan)
sr_tan = sharpe(w_tan, mu, Sigma, rf)

print(pd.Series(w_tan, index=assets).map("{:+.2%}".format).to_string())
print(f"\nreturn {mu_tan:.2%}, volatility {vol_tan:.2%}, Sharpe {sr_tan:.3f}")
excess = mu - rf
print(f"sqrt((mu - rf)' Σ^-1 (mu - rf)) = {np.sqrt(excess @ np.linalg.solve(Sigma, excess)):.3f}")
print("single-asset Sharpe ratios:", np.round(excess / vols, 3))

# Check: maximize the Sharpe ratio numerically
res_sr = minimize(lambda w: -sharpe(w, mu, Sigma, rf), x0=np.full(3, 1 / 3),
                  constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP")
print(f"SLSQP max-Sharpe weights: {np.round(res_sr.x, 4)}")

### The first-order condition as CAPM

In [ ]:
cov_with_tan = Sigma @ w_tan
betas_tan = cov_with_tan / vol_tan**2

check = pd.DataFrame({
    "excess return": mu - rf,
    "beta vs tangency": betas_tan,
    "beta x tangency excess": betas_tan * (mu_tan - rf),
}, index=assets)
check.round(4)

### The capital market line (Example 3)

In [ ]:
a = (m_target - rf) / (mu_tan - rf)
print(f"weight in tangency: {a:.3f} (borrowing {a - 1:.1%} at rf)")
print(f"CML volatility at {m_target:.0%}:      {a * vol_tan:.2%}")
print(f"risky frontier vol at {m_target:.0%}: {frontier_vol(m_target):.2%}")

### How fragile are the tangency weights?

Shift one expected return at a time by $\pm 0.5$ percentage points and recompute.

In [ ]:
rows = {"base": w_tan}
for i, name in enumerate(assets):
    for shift in [-0.005, 0.005]:
        mu_shift = mu.copy()
        mu_shift[i] += shift
        rows[f"{name} {shift * 100:+.1f} pp"] = tangency_weights(mu_shift, Sigma, rf)

sensitivity = pd.DataFrame(rows, index=assets).T
sensitivity.style.format("{:+.1%}")

A half-point change in the expected return of one equity block moves several percentage points
of weight between the two equity blocks. They are 0.8 correlated, so the optimizer treats them as
close substitutes and swings toward whichever looks better. Shifts in the bond return, which
dominates the portfolio, move its weight by about 8 points either way. With many correlated assets the swings become
extreme.

### Sharpe ratio units: monthly vs annual

In [ ]:
periods_per_year = 12
n_months = 600   # 50 years
r = rng.multivariate_normal(mu / periods_per_year, Sigma / periods_per_year, size=n_months)
r_tan = r @ w_tan
rf_monthly = rf / periods_per_year

sr_monthly = (r_tan.mean() - rf_monthly) / r_tan.std(ddof=1)
print(f"monthly Sharpe (sample):  {sr_monthly:.3f}")
print(f"annualized x sqrt(12):    {sr_monthly * np.sqrt(periods_per_year):.3f}")
print(f"true annual Sharpe:       {sr_tan:.3f}")

## 6. Visualizations

In [ ]:
m_grid = np.linspace(0.0, 0.09, 400)
vol_grid = frontier_vol(m_grid)
efficient = m_grid >= B / A

# Long-only frontier by numerical optimization
m_long = np.linspace(mu.min(), mu.max(), 40)
vol_long = []
for m in m_long:
    cons_m = [{"type": "eq", "fun": lambda w: w.sum() - 1},
              {"type": "eq", "fun": lambda w, m=m: w @ mu - m}]
    r_m = minimize(lambda w: w @ Sigma @ w, x0=np.full(3, 1 / 3), constraints=cons_m,
                   bounds=[(0, 1)] * 3, method="SLSQP", options={"ftol": 1e-14})
    vol_long.append(np.sqrt(r_m.fun))
vol_long = np.array(vol_long)

random_w = rng.dirichlet(np.ones(3), size=3_000)
random_mu = random_w @ mu
random_vol = np.sqrt(np.einsum("ij,jk,ik->i", random_w, Sigma, random_w))

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.scatter(random_vol, random_mu, s=4, color="lightgrey", label="random long-only portfolios")
ax.plot(vol_grid[efficient], m_grid[efficient], color="tab:blue", label="efficient frontier (shorts allowed)")
ax.plot(vol_grid[~efficient], m_grid[~efficient], color="tab:blue", linestyle="--", alpha=0.6)
ax.plot(vol_long, m_long, color="tab:orange", label="long-only frontier")

sigma_line = np.linspace(0, 0.18, 100)
ax.plot(sigma_line, rf + sr_tan * sigma_line, color="tab:green", label=f"capital market line (SR = {sr_tan:.2f})")

ax.scatter(vols, mu, color="black", zorder=3)
for name, x, y in zip(assets, vols, mu):
    ax.annotate(name, (x, y), textcoords="offset points", xytext=(6, -4))
ax.scatter(np.sqrt(1 / A), B / A, color="tab:purple", zorder=4, label="GMV")
ax.scatter(vol_tan, mu_tan, color="tab:red", marker="*", s=200, zorder=4, label="tangency")
ax.scatter(0, rf, color="tab:green", zorder=4)
ax.annotate("r_f", (0, rf), textcoords="offset points", xytext=(6, -10))

ax.set_xlim(0, 0.2)
ax.set_xticks(np.arange(0, 0.21, 0.02))
ax.set_ylim(0, 0.09)
ax.set_title("Efficient frontier, tangency portfolio and capital market line")
ax.set_xlabel("annual volatility")
ax.set_ylabel("annual expected return")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.show()

In [ ]:
# Sharpe ratio along the efficient frontier: the maximum is at the tangency portfolio
m_eff = m_grid[efficient]
sr_along = (m_eff - rf) / frontier_vol(m_eff)

plt.figure(figsize=(8, 5))
plt.plot(m_eff, sr_along)
plt.axvline(mu_tan, color="tab:red", linestyle="--", label=f"tangency return {mu_tan:.2%}")
plt.axhline(sr_tan, color="grey", linestyle=":", label=f"max Sharpe {sr_tan:.3f}")
plt.title("Sharpe ratio of frontier portfolios")
plt.xlabel("target expected return m")
plt.ylabel("Sharpe ratio")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
ax = sensitivity.plot(kind="barh", stacked=False, figsize=(9, 5.5), width=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Tangency weights when one expected return moves by ±0.5 pp")
ax.set_xlabel("weight")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.invert_yaxis()
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Two uncorrelated assets: $\mu = (8\%,\ 3\%)$, $\sigma = (20\%,\ 8\%)$, $r_f = 1\%$. Compute the
   tangency weights and Sharpe ratio by hand, then check with `tangency_weights`.
2. A strategy has a monthly Sharpe ratio of 0.15. What is its annualized Sharpe ratio, and
   which assumption does the conversion rely on?

### Intermediate

3. An investor wants 8% annual volatility using the three-asset tangency portfolio and cash.
   How much goes into the tangency portfolio, how much into cash, and what is the expected return?
   Compare with the risky frontier portfolio at 8% volatility.
4. Interpret: a real-estate fund reports a Sharpe ratio of 1.8 from quarterly appraisal-based
   valuations over 8 years. Give two reasons to be sceptical, using this notebook's pitfalls.

### Advanced

5. Show that for uncorrelated assets $SR_{\text{tan}}^2 = \sum_i SR_i^2$. What does this say
   about adding a new strategy with a small positive Sharpe ratio that is uncorrelated with the
   existing portfolio?

## 8. Solutions / Checks

In [ ]:
# 1.
mu_2, vol_2, rf_2 = np.array([0.08, 0.03]), np.array([0.20, 0.08]), 0.01
raw = (mu_2 - rf_2) / vol_2**2
Sigma_2 = np.diag(vol_2**2)
print(f"1. by hand: weights {np.round(raw / raw.sum(), 4)}, Sharpe {np.sqrt(np.sum(((mu_2 - rf_2) / vol_2)**2)):.3f}")
w_2 = tangency_weights(mu_2, Sigma_2, rf_2)
print(f"   formula: weights {np.round(w_2, 4)}, Sharpe {sharpe(w_2, mu_2, Sigma_2, rf_2):.3f}")

# 2.
print(f"2. annual Sharpe = 0.15 * sqrt(12) = {0.15 * np.sqrt(12):.3f}")

# 3. CML: a * vol_tan = 8%
target_vol = 0.08
a_8 = target_vol / vol_tan
print(f"3. tangency {a_8:.1%}, cash {1 - a_8:+.1%}, expected return {rf + a_8 * (mu_tan - rf):.2%}")
# Risky frontier at 8% vol (upper branch): solve A m^2 - 2 B m + C = D sigma^2
m_8 = (B + np.sqrt(B**2 - A * (C - D * target_vol**2))) / A
print(f"   risky frontier at 8% vol: expected return {m_8:.2%}")

**1.** $(\mu_i - r_f)/\sigma_i^2 = 0.07/0.04 = 1.75$ and $0.02/0.0064 = 3.125$: weights
$1.75/4.875 = 35.9\%$ and $64.1\%$. Sharpe $\sqrt{0.35^2 + 0.25^2} = 0.430$.

**2.** $0.15 \cdot \sqrt{12} \approx 0.52$. It assumes monthly returns are independent (no
autocorrelation); with positive autocorrelation the annual volatility is larger than
$\sqrt{12}$ times the monthly one and the annualized Sharpe ratio is overstated.

**3.** $a = 8\% / 5.98\% \approx 1.34$: 134% in the tangency portfolio, borrowing 34% in cash.
Expected return $0.5\% + 1.34 \cdot 2.93\% = 4.42\%$, against $4.26\%$ on the risky frontier at the
same 8% volatility, if borrowing at $r_f$ is possible.

**4.** (a) Appraisal-based valuations are smoothed: reported volatility is far below the true
volatility of the underlying assets, so the Sharpe ratio is inflated. Smoothing also creates
autocorrelation, which breaks $\sqrt{4}$ annualization. (b) 32 quarterly observations give a
very wide confidence interval for any Sharpe ratio (module 06/02). Possibly also: real estate
returns have fat left tails that volatility does not capture.

**5.** With diagonal $\Sigma$,
$SR_{\text{tan}}^2 = (\mu - r_f)^\top\Sigma^{-1}(\mu - r_f) = \sum_i (\mu_i - r_f)^2/\sigma_i^2 = \sum_i SR_i^2$.
Adding an uncorrelated strategy with any positive Sharpe ratio $s$ raises the portfolio's
squared Sharpe ratio by $s^2$. A strategy that is mediocre on its own can still be valuable in
a portfolio if it is uncorrelated with what you already hold; this is the logic behind
multi-strategy funds.

## 9. Conclusion

- Frontier: minimize $\tfrac12 w^\top\Sigma w$ subject to $w^\top\mu = m$, $\mathbf{1}^\top w = 1$; solution $w = \lambda\Sigma^{-1}\mu + \gamma\Sigma^{-1}\mathbf{1}$.
- $\sigma^2(m) = (Am^2 - 2Bm + C)/D$: a hyperbola, leftmost point the GMV; two frontier portfolios span it all.
- With cash, every efficient portfolio holds $w_{\text{tan}} \propto \Sigma^{-1}(\mu - r_f\mathbf{1})$; the CML $\mu_p = r_f + SR_{\text{tan}}\sigma_p$ dominates the risky frontier.
- Max Sharpe: $SR_{\text{tan}}^2 = (\mu - r_f\mathbf{1})^\top\Sigma^{-1}(\mu - r_f\mathbf{1})$; for uncorrelated assets squared Sharpe ratios add.
- The first-order condition reads $\mu_i - r_f = \beta_i(\mu_{\text{tan}} - r_f)$: CAPM, once the tangency portfolio is the market.
- Tangency weights are very sensitive to expected returns: compute with `np.linalg.solve`, and distrust the output.

Next: [05 CAPM and the security market line](./05_capm_and_security_market_line.ipynb).